# Exercise 1 — Introduction, libraries, nearest neighbour

Today we meet the tools we will use for the rest of the semester (NumPy,
Matplotlib, pandas, scikit-learn), we measure a small dataset of leaves
ourselves, and we build the simplest classifier there is: the **nearest
neighbour**. We compute it first on paper, then in a spreadsheet, and only
then in code — so that the code has something to be checked against.

Everything organisational — schedule, assessment, rules for using AI —
is on the course page: <https://tomasvicar.github.io/MLR-public/>.

> **AI assistance is switched off in the first labs.** In Colab go to
> *Tools → Settings → AI Assistance* and uncheck everything. The point is not
> that the tools are bad — from the fifth lab on we use a coding agent on
> purpose — but an algorithm you have never written yourself is an algorithm
> you cannot check.

## Where we write code — IDEs

Two families of tools, and we will use both:

**Standard IDEs** — the code is a `.py` file, it runs from top to bottom.

- **Visual Studio Code** — lightweight, extensible, works with any language;
  this is what we use from the fifth lab on, together with a coding agent.
- **PyCharm** — a full Python IDE, strong debugger and refactoring tools.

**Notebook-style IDEs** — the code is split into cells you run one by one,
and the output stays next to the code.

- **Jupyter Notebook / JupyterLab** — runs locally.
- **Google Colab** — the same thing in the browser, with a free GPU. This is
  what we use in the first labs, so nobody has to install anything.

A notebook is great for exploring data and for teaching, and bad for anything
that has to be reproducible: the cells can be run in any order, so a notebook
that looks fine on screen may not run from top to bottom at all. That is why
real projects live in `.py` files under version control.

# Important libraries

Four libraries carry the whole course. NumPy holds the numbers, Matplotlib
draws them, pandas reads the tables, scikit-learn provides the models we
compare our own implementations against.

## NumPy

- works with **arrays** — vectors, matrices, and higher-dimensional data
- one operation is applied to the whole array at once (no Python `for` loop),
  which is what makes it fast
- a large library of mathematical functions, similar to MATLAB

A dataset in this course is almost always a NumPy array of shape
`(n_samples, n_features)`, so this is the notation we compute in.

In [ ]:
import numpy as np  # standard import

# a list becomes a 1D array (a vector)
first_vector = np.array([1, 2, 3])
print(first_vector)

# a list of lists becomes a 2D array (a matrix)
first_matrix = np.array([[1, 2, 3], [1, 3, 1], [4, 5, 6]])
print(first_matrix)

In [ ]:
# arithmetic is applied element by element to the whole array
print(first_matrix + 5)
print(first_matrix ** 2)

In [ ]:
# shape, slicing and the usual constructors
print(first_matrix.shape)      # (rows, columns)
print(first_matrix[:, 0:2])    # all rows, first two columns
print(np.arange(1, 6))         # 1, 2, 3, 4, 5
print(np.zeros((2, 6)))

In [ ]:
# a comparison gives an array of True/False, which can be used as an index
mask = first_matrix > 1
print(mask)
print(first_matrix[mask])      # only the elements where mask is True

# masking also works on the left-hand side (on a copy, so the original stays)
modified = first_matrix.copy()
modified[modified > 1] = 5
print(modified)

In [ ]:
# careful: * is element-wise, @ is the matrix product
print(first_matrix * first_matrix)
print(first_matrix @ first_matrix)

In [ ]:
# integers stay integers until you ask for something else
print(first_matrix.dtype)
first_matrix_float = first_matrix.astype(np.float32)
print(first_matrix_float.dtype)

Functions you will need today and in the next labs — look them up when you
need them, do not try to remember the list:

`np.sqrt` `np.sum` `np.mean` `np.min` `np.max` `np.argmin` `np.argmax`
`np.unique` `np.sort` `np.argsort` `np.abs` `np.random.rand`

**Try it.** Make an array of the numbers 1 to 10, keep only the even ones and
print their mean. (Expected result: 6.0)

In [ ]:
# TODO: your code here
...

## Matplotlib

- a 2D plotting library, works directly with NumPy arrays
- we use it to *look* at the data before and after every method — a plot
  catches mistakes that a printed number hides

In [ ]:
import matplotlib.pyplot as plt  # standard import
%matplotlib inline

x = np.arange(0, 10)
y = x ** 2

plt.plot(x, y)
plt.xlabel("x")
plt.ylabel("x squared")
plt.show()

In [ ]:
# line and points in one figure, with a title
plt.plot(x, y)
plt.plot(x, y, "r*")
plt.title("Line and points")
plt.show()

In [ ]:
# scatter is the plot we use for datasets: one marker per sample
plt.scatter(x, y)
plt.xlabel("feature 1")
plt.ylabel("feature 2")
plt.show()

In [ ]:
# imshow displays a matrix as an image
plt.imshow(np.random.rand(20, 20))
plt.colorbar()
plt.show()

## Pandas

- works with **tables** (`DataFrame`), the format real data arrives in
- columns have names and may have different types (numbers, text, dates)
- reads and writes CSV, Excel and Google Sheets, and converts to NumPy in one
  call — which is exactly the road our data takes today

In [ ]:
import pandas as pd  # standard import

first_df = pd.DataFrame({"A": [1, 2, 3], "B": [4, 5, 6], "C": ["cat", "dog", "cat"]})
first_df

In [ ]:
print(first_df.dtypes)
print(first_df["A"])          # one column
print(first_df.to_numpy())    # the whole table as a NumPy array

The leaves dataset lives in the course repository, so it can be read straight
from GitHub — the notebook needs no files uploaded to Colab. It contains 518
leaves measured by the students of this course last year.

In [ ]:
DATA = "https://raw.githubusercontent.com/tomasvicar/MLR-public/master/exercises/data/"

leaves_last_year = pd.read_csv(DATA + "leaves.csv")
leaves_last_year.head()

In [ ]:
# iloc selects by position: rows 0-4, columns 0-1
leaves_last_year.iloc[0:5, 0:2]

In [ ]:
# describe gives the summary statistics of every numeric column
leaves_last_year.describe()

In [ ]:
# a condition selects rows, value_counts counts the categories
maples = leaves_last_year[leaves_last_year["Tree type"] == "maple"]
print(maples.shape)
print(leaves_last_year["Tree type"].value_counts())

In [ ]:
leaves_last_year.hist(figsize=(9, 4), bins=40)
plt.show()

**Try it.** What is the mean leaf height of the walnut leaves in
`leaves_last_year`? (One line: select the rows, take the column, take the mean.)

In [ ]:
# TODO: your code here
...

## Scikit-learn

- the classical machine-learning library: models, metrics, preprocessing
- every model has the **same interface** — `fit(X, y)` to train,
  `predict(X)` to use — so models can be swapped by changing one line
- we will keep writing methods ourselves first and then check the result
  against scikit-learn; when the two agree, both are probably right

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# a classic toy dataset: 150 iris flowers, 4 measurements, 3 species
iris = load_iris()
X_iris, y_iris = iris.data, iris.target
print(X_iris.shape, y_iris.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=42)

model = DecisionTreeClassifier(random_state=42)
# try also: LogisticRegression() or SVC() — only this line changes
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Model accuracy: {accuracy_score(y_test, y_pred):.4f}")

## PyTorch

- the deep-learning library we start using in the eighth lab
- its tensors behave much like NumPy arrays, but they also run on the GPU
- it computes derivatives automatically (`autograd`), which is what makes
  training a network a few lines of code instead of a few pages

# Nearest neighbour classifier

The idea, from the first lecture: for a new sample $\mathbf x_\ast$ find the
closest sample in the training set and predict its class,
$\hat y(\mathbf x_\ast) = y_{(1)}$. There is no training phase at all — the
model *is* the data. Everything depends on how we measure "closest"; we use
the Euclidean distance

$$d(\mathbf x, \mathbf x_\ast) = \sqrt{\sum_{j=1}^{D} (x_j - x_{\ast j})^2}
  = \sqrt{(h - h_\ast)^2 + (w - w_\ast)^2}$$

where our two features are the leaf height $h$ and the leaf width $w$.

Playground to get the feeling: <http://vision.stanford.edu/teaching/cs231n-demos/knn/>

## Task 1 — Measure your leaves and build the dataset

We need data, so we make our own. Everybody:

1. takes **three leaves** — one maple, one cherry and one walnut,
2. measures the **height** and the **width** of the leaf blade in centimetres,
   with one decimal place — height and width are the sides of the smallest
   box that contains the blade (see the picture; the small arrows at the
   bottom mark the thickness of the stalk, a feature we use in a later lab),
3. writes the three rows into the **shared spreadsheet**: leaf height, leaf
   width, tree type.

<img src="https://raw.githubusercontent.com/tomasvicar/MLR-public/master/exercises/data/leaf_measurement.png" width="260">

Measuring by hand is the point, not an obstacle: two people measuring the same
leaf get slightly different numbers, and that noise is exactly what a
classifier has to survive.

Paste the id of the shared sheet into `SHEET_ID`. It is the long string in the
URL, `https://docs.google.com/spreadsheets/d/`**`<this part>`**`/edit`.
If you leave it empty, the notebook falls back to last year's 518 leaves and
everything below still works.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SHEET_ID = ""

leaves = None
if SHEET_ID:
    try:
        leaves = pd.read_csv(f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv")
        print(f"{len(leaves)} leaves read from the shared sheet")
    except Exception as error:
        print("the sheet could not be read:", error)
if leaves is None:
    leaves = pd.read_csv(DATA + "leaves.csv")
    print(f"using last year's dataset, {len(leaves)} leaves")

leaves.head()

From the table we take two NumPy arrays: the **features** `X` (one row per
leaf, one column per measurement) and the **labels** `y` (the tree type).
This split is the shape every method in this course expects.

In [ ]:
X = leaves[["Leaf height", "Leaf width"]].to_numpy()
y = leaves["Tree type"].to_numpy()

print(X.shape, y.shape)
print(X[:3], y[:3])

In [ ]:
# one fixed colour per class, used in every plot below
COLORS = {"maple": "tab:green", "cherry": "tab:red", "walnut": "tab:brown"}


def plot_leaves(X, y, title):
    for tree_type in np.unique(y):
        rows = y == tree_type
        plt.scatter(X[rows, 0], X[rows, 1], c=COLORS.get(tree_type, "tab:blue"),
                    label=tree_type, alpha=0.6)
    plt.xlabel("Leaf height [cm]")
    plt.ylabel("Leaf width [cm]")
    plt.title(title)
    plt.legend()


plot_leaves(X, y, "The leaves dataset")
plt.show()

Look at the plot before writing any code. Which two classes overlap? Where
would you draw the boundary by hand? A method can only be as good as what you
can see here.

## Task 2 — Nearest neighbour by hand

Before the computer does it, we do it. Here are six leaves — two of each type
— and one new leaf whose type we do not know:

| # | Leaf height $h$ [cm] | Leaf width $w$ [cm] | Tree type |
| --- | --- | --- | --- |
| 1 | 10 | 14 | maple |
| 2 | 14 | 16 | maple |
| 3 | 8 | 5 | cherry |
| 4 | 10 | 6 | cherry |
| 5 | 16 | 8 | walnut |
| 6 | 18 | 9 | walnut |
| **new** | **15** | **7** | **?** |

**On paper / on the blackboard.** For each of the six leaves compute the
*squared* distance to the new leaf,

$$d_i^2 = (h_i - 15)^2 + (w_i - 7)^2 ,$$

and write the six numbers down. The square root is not needed to find the
minimum — $\sqrt{\cdot}$ is increasing, so the smallest $d^2$ belongs to the
smallest $d$. Take the square root of the winner only, to see the distance in
centimetres. Which leaf is nearest, and what does the classifier predict?

**In a spreadsheet.** Put the same seven rows into Google Sheets or Excel
(height in column `A`, width in column `B`, the new leaf in row 8) and write
one formula in `C2`:

```
=SQRT((A2-$A$8)^2+(B2-$B$8)^2)
```

Drag it down over the six rows. The `$` signs keep the reference to the new
leaf fixed while the rest of the formula moves — this is the spreadsheet
version of a `for` loop, and the numbers must match your paper.

## Task 3 — The same six leaves in NumPy

Now write the code, and check it against the numbers on the blackboard. Four
steps:

1. a function that computes the Euclidean distance between two points,
2. a loop over the training samples that collects the distances,
3. `np.argmin` to find the closest one and read off its class,
4. a plot with the new leaf drawn in the colour of the predicted class.

In [ ]:
small = pd.read_csv(DATA + "leaves_small.csv")
small

In [ ]:
# rows marked "train" are the six known leaves, the row marked "new" is the query
train_small = small[small["Role"] == "train"]
X_small = train_small[["Leaf height", "Leaf width"]].to_numpy()
y_small = train_small["Tree type"].to_numpy()

new_leaf = small[small["Role"] == "new"][["Leaf height", "Leaf width"]].to_numpy()[0]

print(X_small)
print(y_small)
print("new leaf:", new_leaf)

**Step 1.** The distance function. `a` and `b` are two 1D arrays of the same
length; do it with whole-array operations, not with a loop over the features.

In [ ]:
def euk_dist(a, b):
    # TODO: your code here
    return


# a check you can do in your head: the 3-4-5 triangle
assert euk_dist(np.array([0.0, 0.0]), np.array([3.0, 4.0])) == 5.0
print("euk_dist works")

**Step 2 and 3.** Distances to all six training leaves, then the closest one.

In [ ]:
distances = []
for i in range(X_small.shape[0]):
    # TODO: your code here
    distances.append(0.0)
distances = np.array(distances)

# TODO: your code here
index_closest = 0
prediction = y_small[index_closest]

for i in range(X_small.shape[0]):
    print(f"{y_small[i]:>7}  ({X_small[i, 0]:4.1f}, {X_small[i, 1]:4.1f})"
          f"   d^2 = {distances[i] ** 2:6.2f}   d = {distances[i]:5.2f}")
print("prediction:", prediction)

Compare the two columns with the table you filled in on paper. If a single
number differs, the code is wrong — or the blackboard is, and finding out
which is exactly the skill this course is about.

**Step 4.** Plot the six leaves, and add the new one as an `x` in the colour
of the class you predicted — that is the picture of what the classifier
decided.

In [ ]:
plot_leaves(X_small, y_small, "Six leaves and one new")
# TODO: your code here
...
plt.show()

## Task 4 — The whole dataset and two new leaves

The same computation, only the training set is now everything we measured.
Wrap it in a function so it can be called for more than one new leaf, and try
it on two leaves you did not put into the table (or on the two values below).

In [ ]:
def nearest_neighbour(X, y, x_new):
    """Class of the training sample closest to x_new."""
    # TODO: your code here
    return None


# replace with the two leaves you measured
new_leaves = np.array([[12.1, 5.1],
                       [13.0, 15.0]])

predictions = [nearest_neighbour(X, y, x_new) for x_new in new_leaves]
for x_new, predicted in zip(new_leaves, predictions):
    print(f"leaf {x_new} -> {predicted}")

In [ ]:
plot_leaves(X, y, "The whole dataset with two new leaves")
for x_new, predicted in zip(new_leaves, predictions):
    plt.scatter(x_new[0], x_new[1], c=COLORS[predicted], marker="X", s=250,
                edgecolors="black", linewidths=1.5, zorder=3,
                label=f"new leaf -> {predicted}")
plt.legend()
plt.show()

Two questions to answer before you leave — both come back in the next lab:

- The prediction of a nearest-neighbour classifier on a *training* sample is
  always correct. Why, and why does that make training accuracy useless here?
- Leaf height and leaf width are both in centimetres. What would happen if the
  width were recorded in millimetres instead?

# Object-oriented programming — a preview for next time

Python is an object-oriented language and almost everything in it is an
object, with its own data and its own functions:

- **attribute / property** — a variable belonging to an object
- **method** — a function belonging to an object
- **class** — the code describing the type
- **object / instance** — one concrete variable of that type

We need this because a machine-learning model is a natural object: it holds
the training data (attributes) and offers `fit` and `predict` (methods).

In [ ]:
class Adder:
    def __init__(self, value1, value2):
        self.value1 = value1
        self.value2 = value2
        self.value_sum_original = value1 + value2

    def getsum(self):
        return self.value1 + self.value2

    def getsum_timesx(self, x):
        return (self.value1 + self.value2) * x

In [ ]:
adder1 = Adder(2, 3)
adder2 = Adder(4, 3)

print(adder1.value1)
print(adder1.getsum())
print(adder2.getsum())
print(adder1.getsum_timesx(10))

adder1.value1 = 3
print(adder1.getsum())          # 6 — the method recomputes
print(adder1.value_sum_original)  # 5 — the attribute was set in __init__

**Next time, or at home if you are curious.** In the next lab we turn today's
loop into a class with the scikit-learn interface and generalise it from one
neighbour to $k$ neighbours:

```python
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier(n_neighbors=3)
model.fit(X, y)
model.predict(new_leaves)
```

The task will be to write `CustomKNeighborsClassifier` with the same `fit` and
`predict`, and to get the same answers as the line above.